In [ ]:
!sudo apt-get update -y && sudo apt-get install python3.10 python3.10-venv python3.10-dev -y

In [ ]:
!python3.10 -m venv /content/tts_env

In [ ]:
!/content/tts_env/bin/pip install TTS nest_asyncio pyngrok fastapi uvicorn torchcodec

In [ ]:
!/content/tts_env/bin/pip install "transformers<4.50.0"

In [ ]:
%%writefile server.py
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
import wave
import traceback
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from TTS.api import TTS
import uvicorn
from pyngrok import ngrok

#Keep your ngrok auth token inside the secret key option in google colab and give it access to notebook.
secure_ngrok_token = os.environ.get("NGROK_AUTH_TOKEN")
if not secure_ngrok_token:
    raise ValueError("ngrok token is missing from environment variables")

from TTS.tts.configs.xtts_config import XttsConfig
torch.serialization.add_safe_globals([XttsConfig])
_orig_load = torch.load
torch.load = lambda *args, **kwargs: _orig_load(*args, **{**kwargs, "weights_only": False})

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading model...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

app = FastAPI()

class TTSRequest(BaseModel):
    text: str
    chunk_idx: int

def prepend_silence(file_path, duration=0.2):
    try:
        with wave.open(file_path, 'rb') as wav_in:
            params = wav_in.getparams()
            audio_data = wav_in.readframes(params.nframes)
        
        bytes_per_sample = params.sampwidth * params.nchannels
        silence_frames = int(params.framerate * duration)
        silence_data = b'\x00' * (silence_frames * bytes_per_sample)
        
        with wave.open(file_path, 'wb') as wav_out:
            wav_out.setparams(params)
            wav_out.writeframes(silence_data + audio_data)
    except Exception as e:
        print(f"Padding warning: {e}")

@app.post("/synthesize")
def synthesize(data: TTSRequest):
    try:
        print(f"Processing chunk {data.chunk_idx}")
        ref_path = "/content/neil_best.wav"
        output_path = "/content/chunk.wav" # constantly overwritten to prevent file clutter
        
        tts.tts_to_file(
            text=data.text,
            file_path=output_path,
            speaker_wav=ref_path,
            language="en"
        )
        
        padding_duration = 0.6 if data.chunk_idx == 1 else 0.2
        prepend_silence(output_path, duration=padding_duration)
        
        return FileResponse(output_path, media_type="audio/wav")
        
    except Exception as e:
        print("Critical server error")
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

nest_asyncio.apply()
ngrok.set_auth_token(secure_ngrok_token) 
public_url = ngrok.connect(8000)
print(f"🎒 Colab server url: {public_url.public_url}")

uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
import os
import subprocess
from google.colab import userdata

token = userdata.get('NGROK_AUTH_TOKEN')

env = os.environ.copy()
env["NGROK_AUTH_TOKEN"] = token
env["PYTHONUNBUFFERED"] = "1"  # force python to flush logs instantly

print("\nStarting the server...")
process = subprocess.Popen(
    ["/content/tts_env/bin/python", "-u", "server.py"], 
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopping Server...")
    process.terminate()